## Rappresentazione dei circuiti e calcolo dell'equivalenza
Si definisce la classe Circuit in grado di rappresentare circuiti a n qubit e m time-steps come stati quantistici, con associata la classe CircuitShape che gestisce i possibili timesteps dato l'insieme dei gate disponibili ed il numero di qubit

In [1]:
import numpy as np
import csv
from qiskit.circuit import QuantumCircuit
from qiskit.quantum_info import Operator

class CircuitState:
    def __init__(self,shape,steps=None,state=None):  
        if state is None:
            self._shape = shape
            self._state = np.zeros((self._shape.get_n_steps()**len(steps)), dtype=np.csingle)
            index = 0
            for i in range(len(steps)):
                index += self._shape.get_step_index(steps[i]) * self._shape.get_n_steps()**(len(steps)-1-i)
            self._state[index] = 1.
        else:
            self._shape = shape
            self._state = state.copy()
    
    def measure(self):
        probs = np.abs(self._state)**2
        if np.abs(sum(probs) - 1) > 1e-12: raise Exception("The state is not normalized")
        measured_state = np.random.choice(range(len(self._state)), p=probs)
        self._state[:] = 0.
        self._state[measured_state] = 1.
        return self
    
    # converts quantum state to human readable circuit
    # only works if the state is classical
    def get_steps(self):
        if not self.is_classical(): raise Exception("The state is not classical")
        full_index = self._to_base(np.where(self._state != 0)[0][0], self._shape.get_n_steps())
        return [self._shape._steps[i] for i in full_index]
    def _to_base(self,number, base):
        result = np.zeros((self.get_n_steps()),dtype=np.intc)
        i=0
        while number:
            result[i] = number % base
            number //= base
            i+=1
        return result[::-1]


    def get_n_qubits(self):
        return self._shape.get_n_qubits()
    def get_n_steps(self):
        return round(np.emath.logn(self._shape.get_n_steps(), self._state.shape[0]))
    def is_classical(self):
        return len(np.where(self._state != 0)[0]) == 1 
    def get_state(self):
        return self._state.copy()
    

    # saves the circuit in a csv file
    def to_csv(self,filename):
        with open(filename, 'w', newline='') as fout:
            writer = csv.writer(fout, delimiter=",")
            for step in self.get_steps():
                writer.writerow( step )
    #static method to create a circiut from a csv file
    def load_csv(filename,shape):
        with open(filename, 'r', newline='') as fin:
            reader = csv.reader(fin, delimiter=",")
            steps = [row for row in reader]
            return CircuitState(shape,steps=steps)
    load_csv = load_csv
        
    #static method to create a circuit from steps indexes referred to a shape
    def from_steps_ids(ids,shape):
        steps = [shape._steps[i] for i in ids]
        return CircuitState(shape,steps=steps)
    # static method to create a empty circuit n_steps long
    def getID(shape,n_steps):
        return CircuitState(shape,steps=[shape._getID() for i in range(n_steps)])
    
    # returns the corresponding qiskit.circuit.QuantumCircuit
    def to_qiskit(self):
        return CircuitState._to_qiskit_from_steps(self.get_steps())
    
    def _to_qiskit_from_steps(steps):
        n_qubits = len(steps[0])
        circuit = QuantumCircuit(n_qubits)
        for step in steps:
            taken = []
            for i in range(len(step)):
                    if step[i][-1] == "c":
                        for j in range(i+1,len(step)):
                            if j not in taken and step[j][:-1] == step[i][:-1] and step[j][-1] == "t":
                                if step[i][:-1] == "S":
                                    circuit.swap(i,j)
                                elif step[i][:-1] == "CZ":
                                    circuit.cz(i,j)
                                taken.append(j)
                                break #breaks when the gate is close
        return circuit
        

    # draws the circuit using qiskit.circuit.QuantumCircuit.draw() method
    def draw(self,mpl=False):
        if not mpl:
            return self.to_qiskit().draw(output='text')
        else:
            return self.to_qiskit().draw("mpl")
        
    # returns true if the input CircuitState has the same state (assuming the shape is the same)
    def equals(self,circuit):
        return np.array_equal(self._state,circuit._state)
    
    # return true if the circuit is equivalent to ID operator
    def isID(self):
        U = Operator(self.to_qiskit()).to_matrix()
        return np.linalg.norm(U/U[0,0] - np.identity( 2**self.get_n_qubits() )) < 1e-12
    # returns true if c1 and c2 (Cricuit obejcts) implement the same algorithm
    def equivalent(c1,c2):
        U = Operator(c1.to_qiskit().compose(c2.to_qiskit().inverse())).to_matrix()
        return np.linalg.norm(U/U[0,0] - np.identity( 2**c1.get_n_qubits() )) < 1e-12
    equivalent = equivalent
    
    def equivalent_from_steps(steps1,steps2):
        U = Operator(CircuitState._to_qiskit_from_steps(steps1).compose(CircuitState._to_qiskit_from_steps(steps2).inverse())).to_matrix()
        return np.linalg.norm(U/U[0,0] - np.identity( 2**len(steps1[0]) )) < 1e-12


In [454]:
class CircuitShape:
    def __init__(self,n_qubits,gates):
        self._n_qubits = n_qubits
        self._steps = []
        self._find_steps(gates,[],0)
        
    # recursively finds possible timesteps given a set of gates and a number of qubits    
    def _find_steps(self,gates,step,length):
        if length == self._n_qubits:
            self._steps.append(step)
        else:
            for gate_length in gates.keys():
                for gate in gates[gate_length]:
                    if length+gate_length <= self._n_qubits:
                        new_step = step.copy()
                        for block in gate:
                            new_step.append(block)
                        self._find_steps(gates,new_step,length+gate_length)
    
    # returns quantum state from single step list
    def state_of_step(self,step):
        state = np.zeros((len(self._steps)),dtype=np.float64)
        state[self._steps.index(step)] = 1.
        return state
    # return step from single step quantum state
    def step_of_state(self,state):
        return self._steps[np.where(state==1.)[0][0]]
    def index_of_state(self,state):
        return np.where(state==1.)[0][0]
    def get_n_qubits(self):
        return self._n_qubits
    def get_step_index(self,step):
        return self._steps.index(step)
    def get_n_steps(self):
        return len(self._steps)
    def _getID(self):
        return self._steps[0]
    
    def get_steps(self):
        return self._steps.copy()
    
    def print_steps(self,auto_print=True):
        out = ""
        for i in range(len(self._steps)):
            out += str(i) + ": " + str(self._steps[i]) + "\n"
        if auto_print: print(out)
        return out
    
    def _decompose(d,comb,combinations):
        if sum(comb) == d:
            combinations.append(comb)
        else:
            sub = sum(comb)
            if sub == 0: sub=1
            for k in range(2,d+1-sub):
                CircuitShape._decompose(d,comb+[k],combinations)
            
    def _redundant(d,ids1,ids2,equivalences):
        combinations = []
        CircuitShape._decompose(len(ids1),[],combinations)
        for comb in combinations:
            last_term=0
            redundant=True
            for term in comb:
                if (ids1[last_term:term],ids2[last_term:term]) not in equivalences[term-2]:
                    redundant = False
            if redundant==True:
                return True
        return False
            
    def find_equivalences(gates,n_qubits,d_max,dir_out):
        shape = CircuitShape(n_qubits, gates)
        
        equivalences = []
        
        for d in range(2,d_max+1):
            # creates a list of not trivially equivalent circuits of effective d timesteps
            # and a list of id equivalent circuits of effective d timesteps
            actual_circuits = []
            IDequivalent = []
            available_steps = shape.get_steps()
            equivalences.append([])

            # finds all the effectivi circuits with d timesteps
            ids = list(it.product(range(len(available_steps)), repeat=d))
            for i in range(len(ids)):
                c = CircuitState(shape, steps=[available_steps[k] for k in ids[i]])
                if c.get_n_steps() == d and c not in IDequivalent:
                    if c.isID():
                        IDequivalent.append(c)
                    elif all(not c.equals(c1) for c1 in actual_circuits):
                        actual_circuits.append(c)

            # compares the circuits if there's no redundancy
            for i in range(len(actual_circuits)-1):
                for j in range(i+1,len(actual_circuits)):
                    ids1 = [shape.get_step_index(step) for step in actual_circuits[i].get_steps()]
                    ids2 = [shape.get_step_index(step) for step in actual_circuits[j].get_steps()]
                    if not CircuitShape._redundant(d,ids1,ids2,equivalences) and CircuitState.equivalent(actual_circuits[i],actual_circuits[j]):
                        equivalences[d-2].append(ids1,ids2)
            
            # saves the equivalences
            filename = dir_out + "/eq_d=" + str(d) + ".csv"
            with open(filename, 'w', newline='') as fout:
                writer = csv.writer(fout, delimiter="\t")
                writer.writerow(["n_qubits:" +str(n_qubits)])
                writer.writerow(["gates:"])
                for key in gates.keys():
                    writer.writerow(gates[key])
                writer.writerow(["equivalences:"])
                
                for eq in equivalences[d-2]:
                    writer.writerow(eq[0] + eq[1])
            
            filename = dir_out + "/id_d=" + str(d) + ".csv"
            with open(filename, 'w', newline='') as fout:
                writer = csv.writer(fout, delimiter="\t")
                writer.writerow(["n_qubits:" +str(n_qubits)])
                writer.writerow(["gates:"])
                for key in gates.keys():
                    writer.writerow(gates[key])
                writer.writerow(["id equivalences:"])
                for c in IDequivalent:
                    writer.writerow([shape.get_step_index(step) for step in c.get_steps()])

Le due classi una volta scritte e testate vengono aggiunte al modulo QuantumCompiling

In [455]:
#from QuantumCompiling import Circuit, CircuitShape

Test del calcolo dei possibili time-step con la classe CircuitShape

In [456]:
n_qubits = 4
gates = { 
    1 : [ ["ID"]],
    2 : [ ["Sc","St"],["CZc","CZt"] ]
    ,3 : [ ["Sc","ID","St"] ]
    ,4 : [ ["Sc","Sc","St","St"] ] 
}

shape = CircuitShape(n_qubits,gates)
shape.print_steps(); 

0: ['ID', 'ID', 'ID', 'ID']
1: ['ID', 'ID', 'Sc', 'St']
2: ['ID', 'ID', 'CZc', 'CZt']
3: ['ID', 'Sc', 'St', 'ID']
4: ['ID', 'CZc', 'CZt', 'ID']
5: ['ID', 'Sc', 'ID', 'St']
6: ['Sc', 'St', 'ID', 'ID']
7: ['Sc', 'St', 'Sc', 'St']
8: ['Sc', 'St', 'CZc', 'CZt']
9: ['CZc', 'CZt', 'ID', 'ID']
10: ['CZc', 'CZt', 'Sc', 'St']
11: ['CZc', 'CZt', 'CZc', 'CZt']
12: ['Sc', 'ID', 'St', 'ID']
13: ['Sc', 'Sc', 'St', 'St']



Test della classe Circuit: calcolo dello stato dato un circuito human readable e operazione inversa

In [465]:
steps = [
    ['Sc', 'St',"CZc","CZt"],
    ["ID",'Sc', 'St',"ID"],
    ["ID","ID","ID","ID"],
    ['Sc', 'St', 'Sc', 'St']
]

# steps3 = [
#     ["ID","ID","ID"],
#     ["Sc","St","ID"],
#     ["CZc","CZt","ID"],
#     ["ID","CZc","CZt"]
# ]

circuit = CircuitState(shape,steps=steps)

# circuit._state[6] = 0.5
# circuit._state[0] = 0.5

print("Stato:", circuit.get_state())
print("Il circuito è composto da:", circuit.get_n_steps(),"step")
print("Conversione da stato a steps:",circuit.get_steps())
print("Lo stato è classico:",circuit.is_classical())

Stato: [0. 0. 0. ... 0. 0. 0.]
Il circuito è composto da: 4 step
Conversione da stato a steps: [['Sc', 'St', 'CZc', 'CZt'], ['ID', 'Sc', 'St', 'ID'], ['ID', 'ID', 'ID', 'ID'], ['Sc', 'St', 'Sc', 'St']]
Lo stato è classico: True


Test del metodo to qiskit per convertire un oggeto Circuit in un oggetto qiskit.circuit.QuantumCircuit

In [466]:
print(circuit.to_qiskit().draw())
print(CircuitState._to_qiskit_from_steps(steps).draw())

              
q_0: ─X─────X─
      │     │ 
q_1: ─X──X──X─
         │    
q_2: ─■──X──X─
      │     │ 
q_3: ─■─────X─
              
              
q_0: ─X─────X─
      │     │ 
q_1: ─X──X──X─
         │    
q_2: ─■──X──X─
      │     │ 
q_3: ─■─────X─
              


Test dei metodi di caricamento/salvataggio dei circuiti da/su csv

In [459]:
c1 = CircuitState.load_csv("csv/c1.csv",shape)
print("Circuito caricato:", c1.draw())

#salvataggio su csv
c1.to_csv("csv/c1_out.csv")
print("Circuito salvato:", CircuitState.load_csv("csv/c1_out.csv",shape).draw())

Circuito caricato:               
q_0: ─X───────
      │       
q_1: ─X──X────
         │    
q_2: ────X──■─
            │ 
q_3: ───────■─
              
Circuito salvato:               
q_0: ─X───────
      │       
q_1: ─X──X────
         │    
q_2: ────X──■─
            │ 
q_3: ───────■─
              


Test del calcolo dell'equivalenza tra due circuiti: si utilizzano i metodi qiskit per calcolare se due circuiti sono o non sono equivalenti. In particolare si concatena il primo circuito passato come parametro con l'inverso del secondo, si calcola l'operatore U che realizza il circuito complessivo e se ne verifica l'uguaglianza con l'identità

In [467]:
c1 = CircuitState.load_csv("csv/c1.csv",shape)
c2 = CircuitState.load_csv("csv/c2.csv",shape)
c3 = CircuitState.load_csv("csv/c3.csv",shape)

print("Equivalenza c1-c2:",CircuitState.equivalent(c1,c2))
print("Equivalenza c1-c3:",CircuitState.equivalent(c1,c3))

print(c1.draw())
print(c2.draw())
print(c3.draw())

print("Equivalenza from steps c1-c2:",CircuitState.equivalent_from_steps(c1.get_steps(),c2.get_steps()))
print("Equivalenza from steps c1-c3:",CircuitState.equivalent(c1,c3))

Equivalenza c1-c2: True
Equivalenza c1-c3: False
              
q_0: ─X───────
      │       
q_1: ─X──X────
         │    
q_2: ────X──■─
            │ 
q_3: ───────■─
              
              
q_0: ─X───────
      │       
q_1: ─X──X────
         │    
q_2: ────X──■─
            │ 
q_3: ───────■─
              
              
q_0: ─■───────
      │       
q_1: ─■──X──X─
         │  │ 
q_2: ────X──X─
              
q_3: ─────────
              
Equivalenza from steps c1-c2: True
Equivalenza from steps c1-c3: False


Test del metodo per la misura

In [461]:
c1 = CircuitState.load_csv("csv/c1.csv",shape)
c2 = CircuitState.load_csv("csv/c3.csv",shape)
print(c1.draw())
print(c2.draw())
print()

c1._state[np.where(c1._state != 0)[0][0]] = 1/np.sqrt(2)
c1._state[np.where(c2._state != 0)[0][0]] = 1/np.sqrt(2)

try:
    c1.draw()
except Exception as e:
    print(f"Il draw() ha dato errore: {e}")
print()

print("Dopo la misura:",c1.measure().draw())

              
q_0: ─X───────
      │       
q_1: ─X──X────
         │    
q_2: ────X──■─
            │ 
q_3: ───────■─
              
              
q_0: ─■───────
      │       
q_1: ─■──X──X─
         │  │ 
q_2: ────X──X─
              
q_3: ─────────
              

Il draw() ha dato errore: The state is not classical

Dopo la misura:               
q_0: ─X───────
      │       
q_1: ─X──X────
         │    
q_2: ────X──■─
            │ 
q_3: ───────■─
              


In [26]:
n_qubits = 4
gates = [
    ["ID"],
    ["Sc","St"],["CZc","CZt"],
    ["Sc","ID","St"] ,
    ["Sc","Sc","St","St"] 
]

gates_lists = [[g for g in gates if len(g)==i] for i in range(1,max([len(g) for g in gates])+1)]


In [33]:
import scipy.sparse as sp
id = sp.identity(10, format='csr')

type(id)

scipy.sparse._csr.csr_matrix